# 📖🤖 Talk to a Book
### Build an AI agent that reads a novel and chats with you about it

By the end of this notebook you will have an agent that:
- **reads a whole novel** you choose (any `.txt` or `.pdf`),
- **searches it** to answer questions about characters, events, chapters, relationships, themes and the ending,
- **reasons in several steps**: it decides for itself when to search, what to search for and when it knows enough,
- **remembers the conversation**, so follow-up questions like *"why did she do that?"* just work,
- is **fully traced** so you can watch every step it takes.

| Library | What it does for us |
|---|---|
| **LangChain** | Building blocks: documents, text splitter, embeddings, vector store, chat model, tools |
| **LangGraph** | The agent itself: a small graph where the model can loop between *thinking* and *using tools*, plus memory |
| **LangSmith** | Observability: a trace of every LLM call and tool call, plus a mini evaluation |

```
 novel.txt ─► chapters ─► chunks ─► embeddings ─► vector store
                                                     ▲
                                                     │ search_book / read_chapter (tools)
 you ◄──► ┌──────── LangGraph agent ────────┐        │
          │  agent (Gemini) ⇄ tools ────────┼────────┘
          │  memory: checkpointer + thread  │
          └─────────────────────────────────┘ ──► every step traced in LangSmith
```

| Step | Time |
|---|---|
| 0. Setup | 10 min |
| 1-2. Load the book, split into chapters and chunks | 20 min |
| 3. Embeddings and a vector store | 15 min |
| 4-5. Tools and the chat model | 15 min |
| 6. Build the agent with LangGraph | 20 min |
| 7. Multi-turn chat with memory | 15 min |
| 8. LangSmith: traces and evaluation | 10 min |

## Step 0 · Setup

1. Install the packages (once, in a terminal, ideally inside a virtual environment):
   `pip install -r requirements.txt`
2. Get three API keys:
   - **Gemini** (the LLM): https://aistudio.google.com/apikey
   - **Voyage AI** (embeddings): https://dash.voyageai.com. The free tier works (the notebook respects its
     3 requests/minute limit). Adding a payment method only makes Step 3 faster; the free tokens still apply.
   - **LangSmith** (tracing): https://smith.langchain.com → Settings → API Keys
3. Put them in a `.env` file next to this notebook (copy `.env.example`), or just paste them when the next cell asks.

✏️ **Choose your book** in the cell below. Any plain-text (`.txt`) or `.pdf` novel works, and free classics are at
[gutenberg.org](https://www.gutenberg.org) (download "Plain Text UTF-8").

In [ ]:
BOOK_PATH = "books/pride_and_prejudice.txt"   # ✏️ path to the novel you want to talk to (.txt or .pdf)
BOOK_TITLE = "Pride and Prejudice"            # ✏️ its title (used in the agent's instructions)

CHAT_MODEL = "google_genai:gemini-3.5-flash-lite"   # "provider:model"; any LangChain chat model works
EMBEDDING_MODEL = "voyage-4"

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()  # read keys from a .env file, if there is one

for key in ["GOOGLE_API_KEY", "VOYAGE_API_KEY", "LANGSMITH_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(f"Paste your {key}: ")

# LangSmith: with tracing switched on, every LangChain / LangGraph call is recorded automatically.
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "talk-to-a-book")
print("Ready! Traces will appear in the LangSmith project:", os.environ["LANGSMITH_PROJECT"])

## Step 1 · Load the book

A book is just a (long) string. Books from Project Gutenberg start and end with a licence, so we keep only the
text between the `*** START OF …` and `*** END OF …` markers.

In [ ]:
import re
from pathlib import Path

def load_text(path: str) -> str:
    path = Path(path)
    if path.suffix.lower() == ".pdf":
        from pypdf import PdfReader
        return "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
    return path.read_text(encoding="utf-8", errors="ignore")

text = load_text(BOOK_PATH)

start = re.search(r"\*\*\* ?START OF.*?\*\*\*", text)
end = re.search(r"\*\*\* ?END OF.*?\*\*\*", text)
text = text[start.end() if start else 0 : end.start() if end else len(text)]

print(f"{len(text):,} characters, about {len(text.split()):,} words")
print(text[:600])

## Step 2 · Split into chapters, then into chunks

**Chapters** matter: if we know which chapter every piece of text came from, the agent can answer
*"what happens in chapter 12?"* and cite its sources as `[Ch. 12]`.

We find chapter headings with a regular expression. A heading is a line like `CHAPTER XII.` or `Chapter 3`.
The text before the first heading (preface, contents, the picture captions you saw above) is dropped, and so are
tiny pieces (table-of-contents lines). Real-world data is messy, and cleaning it is part of building a good RAG system.

In [ ]:
CHAPTER_PATTERN = r"^\s*(?:CHAPTER|Chapter)\s+[0-9IVXLC]+\b.*$"   # ✏️ adjust if your book's headings look different

parts = re.split(CHAPTER_PATTERN, text, flags=re.MULTILINE)
chapters = [part.strip() for part in parts[1:] if len(part.split()) > 100]
chapters = [re.sub(r"\[Illustration.*?\]\]?", "", ch, flags=re.DOTALL) for ch in chapters]  # drop picture captions
if not chapters:          # no headings found? then treat the whole book as a single chapter
    chapters = [text]

print(f"Found {len(chapters)} chapters. Chapter 1 begins:\n")
print(chapters[0][:400])

✏️ **Check:** does the number of chapters match your book? (*Pride and Prejudice* has 61.) If not, look at how
the headings are written in the text file and adjust `CHAPTER_PATTERN`.

Next, the **chunks**. An LLM can't read the whole novel for every question (too slow and expensive), so we cut
each chapter into overlapping chunks of about 1,200 characters. LangChain's text splitter turns them into
`Document` objects, and each one remembers its chapter number in `metadata`.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
docs = splitter.create_documents(
    chapters,
    metadatas=[{"chapter": number} for number in range(1, len(chapters) + 1)],
)
print(f"{len(docs)} chunks")
docs[100]

✏️ **Try it:** change `chunk_size` to 400 and then 4000. How many chunks do you get? Small chunks give precise
matches but little context; big chunks give more context but blurrier matches.
(Set it back to 1200 before moving on.)

## Step 3 · Embeddings and a vector store

An **embedding model** turns text into a list of numbers (a vector) so that texts with *similar meaning* get
*similar vectors*. We embed every chunk once and keep the vectors in a **vector store**. To answer a question we
embed the question and fetch the closest chunks. That is the **R** (retrieval) in **RAG**.

**Free-tier friendly:** without a payment method, Voyage allows only **3 requests and 10,000 tokens per minute**,
and a novel is ~170,000 tokens. `FreeTierVoyageEmbeddings` (in `voyage_free_tier.py`) is a drop-in replacement for
`VoyageAIEmbeddings` that sends small requests (at most 3,000 tokens) and **waits whenever the next request would break
the limits**. Embedding a whole novel therefore takes **about 20 minutes**. Progress is saved after every request, so
you can stop the cell and re-run it later: it continues where it left off.

💡 Start this cell, then keep reading and writing Steps 4-6 while it works (you just can't *run* them yet).
The index is saved to `index/`, so you only ever wait once per book.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from voyage_free_tier import FreeTierVoyageEmbeddings

embeddings = FreeTierVoyageEmbeddings(model=EMBEDDING_MODEL)   # free_tier=False if you added a payment method

for number, doc in enumerate(docs):
    doc.id = f"chunk-{number}"          # stable ids, so an interrupted run can resume

index_file = Path("index") / f"{Path(BOOK_PATH).stem}.json"
index_file.parent.mkdir(exist_ok=True)
if index_file.exists():
    vector_store = InMemoryVectorStore.load(str(index_file), embeddings)
else:
    vector_store = InMemoryVectorStore(embeddings)

todo = [doc for doc in docs if doc.id not in vector_store.store]
batches = embeddings.batches(todo)      # groups of chunks that fit in one request (<= 3,000 tokens)
print(f"{len(docs) - len(todo)} chunks already embedded, {len(todo)} to go in {len(batches)} requests")
for batch in batches:
    vector_store.add_documents(batch)
    vector_store.dump(str(index_file))  # save progress after every request
    print(f"Embedded {len(vector_store.store)} / {len(docs)} chunks")
print("Index ready:", index_file)

In [ ]:
results = vector_store.similarity_search("Mr. Darcy proposes to Elizabeth", k=3)
for doc in results:
    print(f"--- Chapter {doc.metadata['chapter']} ---")
    print(doc.page_content[:300], "\n")

✏️ **Try it:** search for a *feeling* rather than a name, e.g. *"she realises she has misjudged him"*.
Embeddings match meaning, not just words. (Each search is one small Voyage request, so on the free tier a search
may pause for a few seconds to respect the 3-requests-per-minute limit.)

## Step 4 · Tools: how the agent reads the book

A **tool** is a normal Python function the LLM is allowed to call. The `@tool` decorator turns it into something
the model understands, and **the docstring is the tool's instruction manual**, so write it carefully!

Our agent gets two tools:
- `search_book`: semantic search over the chunks (good for *who / why / how* questions),
- `read_chapter`: the full text of one chapter (good for *"what happens in chapter 5?"* or *"how does it end?"*).

In [ ]:
from langchain_core.tools import tool

@tool
def search_book(query: str) -> str:
    """Search the novel for passages relevant to the query. Use specific names, places and events.
    Returns the 6 most relevant passages, each labelled with its chapter number, in story order."""
    results = vector_store.similarity_search(query, k=6)
    results.sort(key=lambda doc: doc.metadata["chapter"])
    return "\n\n".join(f"[Chapter {doc.metadata['chapter']}]\n{doc.page_content}" for doc in results)

@tool
def read_chapter(chapter: int) -> str:
    """Return the full text of one chapter, given its number. Use it for questions about a specific
    chapter, or read the last chapter to find out how the book ends."""
    if not 1 <= chapter <= len(chapters):
        return f"There is no chapter {chapter}. The book has chapters 1 to {len(chapters)}."
    return chapters[chapter - 1]

tools = [search_book, read_chapter]
print(search_book.invoke({"query": "Mr. Collins proposes"})[:500])

## Step 5 · The chat model

`init_chat_model` creates a chat model from a `"provider:model"` string. Swapping Gemini for another provider is
a one-line change. `bind_tools` tells the model which tools exist, and the model then decides by itself when to use them.

In [ ]:
import logging
from langchain.chat_models import init_chat_model

logging.getLogger("google_genai.models").setLevel(logging.ERROR)   # hide a harmless internal notice from Google's SDK

llm = init_chat_model(CHAT_MODEL)
llm_with_tools = llm.bind_tools(tools)

print(llm.invoke("In one sentence: what is a novel?").text)

## Step 6 · Build the agent with LangGraph

An agent is a loop: the model **thinks** → maybe **uses a tool** → looks at the result → thinks again → … →
**answers**. In LangGraph we draw that loop as a graph:

- **State**: what flows through the graph. `MessagesState` is simply the list of chat messages.
- **Nodes**: functions that update the state. `agent` calls the LLM, and `tools` runs whatever tools the LLM asked for.
- **Edges**: what runs next. After `agent`, a *conditional edge* goes to `tools` if the LLM asked for a tool, or
  ends if it wrote an answer. After `tools` we always go back to `agent`.
- **Checkpointer**: saves the state after every step, per conversation (`thread_id`). This is the agent's **memory**.

In [ ]:
SYSTEM_PROMPT = f"""You are a friendly, insightful guide to the novel "{BOOK_TITLE}" ({len(chapters)} chapters).
Answer the reader's questions about its characters, events, chapters, relationships, themes and ending.

- Before answering any question about the book, use your tools to find evidence in it. Don't rely on memory.
- For questions about how something changes over the story, search more than once (early, middle and late events).
- Cite the chapters you used like [Ch. 12].
- If the book doesn't answer the question, say so."""

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

def agent(state: MessagesState):
    """Read the conversation so far, then either ask for tools or write the answer."""
    response = llm_with_tools.invoke([SystemMessage(SYSTEM_PROMPT)] + state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)   # tool calls? -> "tools", otherwise -> END
builder.add_edge("tools", "agent")                        # after using tools, think again

graph = builder.compile(checkpointer=InMemorySaver())

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))   # drawn by the mermaid.ink web service
except Exception:
    print(graph.get_graph().draw_mermaid())                # offline? paste this into https://mermaid.live

💡 LangChain can build this exact graph in one line: `create_agent(llm, tools, system_prompt=SYSTEM_PROMPT, checkpointer=InMemorySaver())`
(from `langchain.agents`). We built it by hand so you can see how it works inside.

## Step 7 · Chat with the book, with memory

`ask()` sends one question into the graph. We *stream* the steps so you can see the agent's multi-step
reasoning: each 🔧 line is a tool call the agent decided to make.

The `thread_id` names the conversation. Same `thread_id` = the agent remembers everything said before.
`@traceable` makes each question show up as one neat trace in LangSmith.

In [ ]:
import uuid
from IPython.display import Markdown
from langsmith import traceable

@traceable(name="ask_the_book")
def ask(question: str, thread_id: str = "reader-1"):
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 20}
    for step in graph.stream({"messages": [("user", question)]}, config, stream_mode="updates"):
        for node, update in step.items():
            if node != "agent":
                continue
            message = update["messages"][-1]
            for call in message.tool_calls:                # the agent decided to use a tool
                print(f"  🔧 {call['name']}({call['args']})")
            if not message.tool_calls:                     # no tool call means this is the answer
                display(Markdown(message.text))
                reason = message.response_metadata.get("finish_reason")
                if reason not in (None, "STOP"):           # e.g. Gemini's safety filter cut the answer short
                    print(f"  ⚠️ Gemini stopped early ({reason}), so the answer may be incomplete. Try rephrasing.")

ask("Who is Mr. Darcy, and what do people think of him at first?")

In [ ]:
ask("How does Elizabeth's opinion of him change over the course of the novel?")

Now a **follow-up**. *"Her"* and *"that"* only make sense because the agent remembers this conversation:

In [ ]:
ask("What was the turning point for her?")

**Is it really memory?** Ask *about the conversation itself* in two threads. `reader-1` remembers everything, while
`someone-else` is a brand-new thread with nothing to remember. (A follow-up like *"the turning point for her"* can
sometimes be *guessed* from the book alone, but a question about the conversation can't.)

In [ ]:
ask("Remind me: what have we talked about so far?")

In [ ]:
ask("Remind me: what have we talked about so far?", thread_id="someone-else")

The memory is just the saved graph state. Let's look inside the conversation `reader-1`:

In [ ]:
state = graph.get_state({"configurable": {"thread_id": "reader-1"}})
for message in state.values["messages"]:
    calls = [call["name"] for call in getattr(message, "tool_calls", [])]
    print(f"{message.type:>6}: {message.text[:80]!r}" + (f"  -> calls {calls}" if calls else ""))

### 💬 Your turn: chat freely

Ideas: *"What happens in chapter 34?"* · *"Who is Mr. Wickham?"* → *"Was he telling the truth?"* ·
*"What are the main themes?"* · *"How does the book end?"*. Type `quit` to stop.

In [ ]:
thread_id = str(uuid.uuid4())   # a fresh conversation
print("Chat with the book! Type 'quit' to stop.")
while True:
    question = input("You: ")
    if question.strip().lower() in {"", "quit", "exit"}:
        break
    ask(question, thread_id)

## Step 8 · LangSmith: see (and measure) what the agent does

Open https://smith.langchain.com → **Projects** → **talk-to-a-book**. Click an `ask_the_book` trace and explore:

- the **graph steps**: `agent` → `tools` → `agent` → …, which is the multi-step reasoning you saw above,
- the **exact prompt** sent to Gemini (system prompt + conversation history = the memory!),
- each **tool call**: its input (the search query the agent chose) and output (the passages it read),
- **tokens and latency** for every step.

### Bonus: a tiny evaluation

"It looked right on 3 questions" isn't a test. LangSmith can store a **dataset** of questions with reference
answers, run the agent on all of them, and grade the answers with an **LLM-as-a-judge**.
The reference answers below are for *Pride and Prejudice*, so change them if you picked another book.

In [ ]:
from langsmith import Client
from pydantic import BaseModel, Field

client = Client()
DATASET = "talk-to-a-book-demo"

examples = [
    {"inputs": {"question": "Who does Elizabeth Bennet marry at the end of the novel?"},
     "outputs": {"answer": "Mr. Fitzwilliam Darcy."}},
    {"inputs": {"question": "Why does Charlotte Lucas marry Mr. Collins?"},
     "outputs": {"answer": "For financial security and a comfortable home rather than for love."}},
    {"inputs": {"question": "What happens in chapter 34?"},
     "outputs": {"answer": "Darcy proposes to Elizabeth for the first time and she angrily refuses him."}},
]
if not client.has_dataset(dataset_name=DATASET):
    dataset = client.create_dataset(DATASET)
    client.create_examples(dataset_id=dataset.id, examples=examples)

In [ ]:
def run_agent(inputs: dict) -> dict:
    """What we evaluate: answer one question in a fresh conversation."""
    config = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 20}
    result = graph.invoke({"messages": [("user", inputs["question"])]}, config)
    return {"answer": result["messages"][-1].text}

class Grade(BaseModel):
    correct: bool = Field(description="Does the answer agree with the reference answer?")

judge = llm.with_structured_output(Grade)   # an LLM-as-a-judge that must reply with a Grade object

def correct(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    grade = judge.invoke(
        f"Question: {inputs['question']}\n"
        f"Reference answer: {reference_outputs['answer']}\n"
        f"Answer to grade: {outputs['answer']}"
    )
    return grade.correct

results = client.evaluate(run_agent, data=DATASET, evaluators=[correct], experiment_prefix="book-agent")

Open the experiment link printed above: you'll see every question, the agent's answer, the judge's verdict and
the full trace behind each answer. Change something (the system prompt, `k` in `search_book`, the chunk size, the
model) and run the evaluation again to *measure* whether it got better.

---

## 🎉 You built a RAG agent! Where to go next

- **Talk to another book:** change `BOOK_PATH` and `BOOK_TITLE` in Step 0 and re-run everything.
- **Spoiler guard:** add a `max_chapter` setting and filter the search with
  `vector_store.similarity_search(query, k=6, filter=lambda doc: doc.metadata["chapter"] <= max_chapter)`.
- **Memory that survives a restart:** swap `InMemorySaver` for `SqliteSaver` (package `langgraph-checkpoint-sqlite`).
- **Long conversations:** trim or summarize old messages before calling the LLM (`langchain_core.messages.trim_messages`).
- **Better retrieval:** combine keyword search (BM25) with embeddings, or add a reranker.
- **A web UI:** wrap `ask()` in a small Streamlit or Gradio app.